In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 4, Finished, Available, Finished, False)

In [3]:
landing_path = "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/trucks.csv"

df_trucks = (
    spark.read
    .option("header", "true")
    .csv(landing_path)
)

display(df_trucks)

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8fe385bd-eedc-4fb5-b1d0-1240c0cfd842)

In [4]:
df_trucks.printSchema()

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 6, Finished, Available, Finished, False)

root
 |-- truck_id: string (nullable = true)
 |-- unit_number: string (nullable = true)
 |-- make: string (nullable = true)
 |-- model_year: string (nullable = true)
 |-- vin: string (nullable = true)
 |-- acquisition_date: string (nullable = true)
 |-- acquisition_mileage: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- tank_capacity_gallons: string (nullable = true)
 |-- status: string (nullable = true)
 |-- home_terminal: string (nullable = true)



In [5]:
print(f"Source records: {df_trucks.count()}")

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 7, Finished, Available, Finished, False)

Source records: 120


In [6]:
truck_schema = StructType([
    StructField("truck_id", StringType(), True),
    StructField("unit_number", StringType(), True),
    StructField("make", StringType(), True),
    StructField("model_year", IntegerType(), True),
    StructField("vin", StringType(), True),
    StructField("acquisition_date", DateType(), True),
    StructField("acquisition_mileage", IntegerType(), True),
    StructField("fuel_type", StringType(), True),
    StructField("tank_capacity_gallons", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("home_terminal", StringType(), True)
])

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 8, Finished, Available, Finished, False)

In [7]:
df_trucks = (
    spark.read
    .option("header", "true")
    .schema(truck_schema)
    .csv(landing_path)
)

display(df_trucks)

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a33ab550-cb11-4a12-8e9e-6d9330a54fd8)

In [8]:
df_trucks.printSchema()

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 10, Finished, Available, Finished, False)

root
 |-- truck_id: string (nullable = true)
 |-- unit_number: string (nullable = true)
 |-- make: string (nullable = true)
 |-- model_year: integer (nullable = true)
 |-- vin: string (nullable = true)
 |-- acquisition_date: date (nullable = true)
 |-- acquisition_mileage: integer (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- tank_capacity_gallons: double (nullable = true)
 |-- status: string (nullable = true)
 |-- home_terminal: string (nullable = true)



In [9]:
source_count = df_trucks.count()
print(f"Source records: {source_count}")

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 11, Finished, Available, Finished, False)

Source records: 120


In [10]:
null_truck_ids = (
    df_trucks
    .filter(F.col("truck_id").isNull())
    .count()
)

print(f"NULL truck IDs: {null_truck_ids}")

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 12, Finished, Available, Finished, False)

NULL truck IDs: 0


In [11]:

duplicate_truck_ids = (
    df_trucks
    .groupBy("truck_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate truck IDs: {duplicate_truck_ids}")

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 13, Finished, Available, Finished, False)

Duplicate truck IDs: 0


In [12]:

invalid_model_year = (
    df_trucks
    .filter(
        (F.col("model_year") < 1900) |
        (F.col("model_year") > 2100)
    )
    .count()
)

print(f"Invalid model years: {invalid_model_year}")

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 14, Finished, Available, Finished, False)

Invalid model years: 0


In [13]:
negative_mileage = (
    df_trucks
    .filter(F.col("acquisition_mileage") < 0)
    .count()
)

print(f"Negative acquisition mileage: {negative_mileage}")

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 15, Finished, Available, Finished, False)

Negative acquisition mileage: 0


In [14]:


invalid_tank_capacity = (
    df_trucks
    .filter(F.col("tank_capacity_gallons") <= 0)
    .count()
)

print(f"Invalid tank capacity: {invalid_tank_capacity}")

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 16, Finished, Available, Finished, False)

Invalid tank capacity: 0


In [15]:
df_trucks = (
    df_trucks
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("trucks.csv"))
)

df_trucks.createOrReplaceTempView("trucks_source")

print("Truck source data prepared for Bronze layer.")

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 17, Finished, Available, Finished, False)

Truck source data prepared for Bronze layer.


In [16]:
bronze_trucks_path = "Tables/dbo/bronze_trucks"

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_trucks (
    truck_id STRING,
    unit_number STRING,
    make STRING,
    model_year INT,
    vin STRING,
    acquisition_date DATE,
    acquisition_mileage INT,
    fuel_type STRING,
    tank_capacity_gallons DOUBLE,
    status STRING,
    home_terminal STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")

print("bronze_trucks table is ready.")

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 18, Finished, Available, Finished, False)

bronze_trucks table is ready.


In [17]:
result = spark.sql("""
MERGE INTO bronze_trucks AS target
USING trucks_source AS source

ON target.truck_id = source.truck_id

WHEN MATCHED THEN
    UPDATE SET
        target.unit_number = source.unit_number,
        target.make = source.make,
        target.model_year = source.model_year,
        target.vin = source.vin,
        target.acquisition_date = source.acquisition_date,
        target.acquisition_mileage = source.acquisition_mileage,
        target.fuel_type = source.fuel_type,
        target.tank_capacity_gallons = source.tank_capacity_gallons,
        target.status = source.status,
        target.home_terminal = source.home_terminal,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        truck_id,
        unit_number,
        make,
        model_year,
        vin,
        acquisition_date,
        acquisition_mileage,
        fuel_type,
        tank_capacity_gallons,
        status,
        home_terminal,
        ingestion_timestamp,
        source_file
    )
    VALUES (
        source.truck_id,
        source.unit_number,
        source.make,
        source.model_year,
        source.vin,
        source.acquisition_date,
        source.acquisition_mileage,
        source.fuel_type,
        source.tank_capacity_gallons,
        source.status,
        source.home_terminal,
        source.ingestion_timestamp,
        source.source_file
    )
""")

display(result)

print("Truck Bronze MERGE completed successfully.")

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc8dfdd1-947f-4739-9616-2919ef805b06)

Truck Bronze MERGE completed successfully.


In [18]:
bronze_count = spark.sql("""
SELECT COUNT(*) AS count
FROM bronze_trucks
""").collect()[0]["count"]

print(f"Bronze truck records: {bronze_count}")

display(
    spark.sql("""
    SELECT *
    FROM bronze_trucks
    LIMIT 10
    """)
)

StatementMeta(, a95784f2-33d7-4b9f-9f4c-b92ef2550d07, 20, Finished, Available, Finished, False)

Bronze truck records: 120


SynapseWidget(Synapse.DataFrame, 8f0168fc-d9c7-40a1-a09e-5c50b5c5b4f9)